# Performance Scaling Analysis - Interactive Dashboard

This notebook provides interactive visualization and analysis of transformer model performance benchmarks across different sequence lengths and model sizes.

## 1. Load and Explore Performance Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Load CSV
csv_path = Path('perf_summary.csv')
df = pd.read_csv(csv_path)

print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn Names:")
print(df.columns.tolist())
print("\nData Types:")
print(df.dtypes)
print("\nBasic Statistics:")
print(df[['seq_len', 'mflops', 'total_cpu_time_s', 'total_mem_mb']].describe())

## 2. Data Cleaning and Preprocessing

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Clean numeric columns
numeric_cols = ['seq_len', 'mflops', 'total_cpu_time_s', 'total_mem_mb']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop any remaining NaNs
df = df.dropna(subset=numeric_cols)

# Extract top ops into separate dataframe for easier analysis
top_ops_cols = [col for col in df.columns if col.startswith('top') and col.endswith('_op')]

print(f"\nCleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Models present: {df['model_size'].unique()}")
print(f"Sequence lengths: {sorted(df['seq_len'].unique())}")

## 3. Compute Throughput (MFLOPs) Scaling

In [ ]:
# MFLOPs vs Sequence Length
fig, ax = plt.subplots(figsize=(12, 6))

for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax.plot(model_data['seq_len'], model_data['mflops'], 
           marker='o', linewidth=2.5, markersize=8, label=f'Llama {model}')

ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
ax.set_ylabel('MFLOPs (Million FLOPs)', fontsize=12, fontweight='bold')
ax.set_title('Compute Throughput Scaling: MFLOPs vs Sequence Length', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

# Summary statistics
print("\nMFLOPs Scaling Summary:")
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    min_mflops = model_data['mflops'].min()
    max_mflops = model_data['mflops'].max()
    scaling = max_mflops / min_mflops
    print(f"  Llama {model}: {min_mflops:.0f} → {max_mflops:.0f} MFLOPs ({scaling:.1f}x)")

## 4. Latency Scaling (CPU Time)

In [ ]:
# CPU Time vs Sequence Length
fig, ax = plt.subplots(figsize=(12, 6))

for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax.plot(model_data['seq_len'], model_data['total_cpu_time_s'], 
           marker='s', linewidth=2.5, markersize=8, label=f'Llama {model}')

ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
ax.set_ylabel('CPU Time (seconds)', fontsize=12, fontweight='bold')
ax.set_title('Latency Scaling: CPU Time vs Sequence Length', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\nCPU Time Scaling Summary:")
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    min_time = model_data['total_cpu_time_s'].min()
    max_time = model_data['total_cpu_time_s'].max()
    scaling = max_time / min_time
    print(f"  Llama {model}: {min_time:.4f}s → {max_time:.4f}s ({scaling:.1f}x)")

## 5. Memory Usage Scaling

In [ ]:
# Memory vs Sequence Length
fig, ax = plt.subplots(figsize=(12, 6))

for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax.plot(model_data['seq_len'], model_data['total_mem_mb'], 
           marker='^', linewidth=2.5, markersize=8, label=f'Llama {model}')

ax.set_xlabel('Sequence Length', fontsize=12, fontweight='bold')
ax.set_ylabel('Peak Memory (MB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Usage Scaling: Peak Memory vs Sequence Length', 
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=11, loc='best')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("\nMemory Usage Scaling Summary:")
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    min_mem = model_data['total_mem_mb'].min()
    max_mem = model_data['total_mem_mb'].max()
    scaling = max_mem / min_mem
    print(f"  Llama {model}: {min_mem:.1f}MB → {max_mem:.1f}MB ({scaling:.1f}x)")

## 6. Efficiency Metrics (MFLOPs/sec/MB)

In [ ]:
# Compute efficiency metrics
df['mflops_per_sec'] = df['mflops'] / df['total_cpu_time_s']
df['mflops_per_mb'] = df['mflops'] / df['total_mem_mb']

# Plot MFLOPs/sec (compute efficiency)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: MFLOPs/sec
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax1.plot(model_data['seq_len'], model_data['mflops_per_sec'], 
            marker='o', linewidth=2.5, markersize=8, label=f'Llama {model}')

ax1.set_xlabel('Sequence Length', fontsize=11, fontweight='bold')
ax1.set_ylabel('MFLOPs/sec', fontsize=11, fontweight='bold')
ax1.set_title('Compute Efficiency: MFLOPs/sec', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xscale('log')

# Right: MFLOPs/MB (memory efficiency)
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax2.plot(model_data['seq_len'], model_data['mflops_per_mb'], 
            marker='^', linewidth=2.5, markersize=8, label=f'Llama {model}')

ax2.set_xlabel('Sequence Length', fontsize=11, fontweight='bold')
ax2.set_ylabel('MFLOPs/MB', fontsize=11, fontweight='bold')
ax2.set_title('Memory Efficiency: MFLOPs/MB', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xscale('log')

plt.tight_layout()
plt.show()

print("\nEfficiency Summary:")
print(df[['model_size', 'seq_len', 'mflops_per_sec', 'mflops_per_mb']].to_string(index=False))

## 7. Top Operations Breakdown

## 8. Comparative Analysis: Model vs Model

In [ ]:
# Create comparison heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = [
    ('mflops', 'MFLOPs', axes[0]),
    ('total_cpu_time_s', 'CPU Time (s)', axes[1]),
    ('total_mem_mb', 'Memory (MB)', axes[2])
]

for metric, title, ax in metrics:
    pivot_data = df.pivot(index='model_size', columns='seq_len', values=metric)
    pivot_data = pivot_data.sort_index()
    sns.heatmap(pivot_data, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, cbar_kws={'label': title})
    ax.set_title(f'{title} Heatmap', fontweight='bold')
    ax.set_ylabel('Model Size')
    ax.set_xlabel('Sequence Length')

plt.tight_layout()
plt.show()

## 9. Key Insights Summary

In [ ]:
print("="*70)
print("PERFORMANCE SCALING INSIGHTS")
print("="*70)

print("\n1. COMPUTE SCALING (MFLOPs):")
for model in sorted(df['model_size'].unique()):
    model_df = df[df['model_size'] == model]
    avg_mflops = model_df['mflops'].mean()
    print(f"   Llama {model}: ~{avg_mflops:.0f} MFLOPs average")

print("\n2. SCALING BEHAVIOR:")
for model in sorted(df['model_size'].unique()):
    model_df = df[df['model_size'] == model].sort_values('seq_len')
    seq_increase = (model_df['seq_len'].iloc[-1] / model_df['seq_len'].iloc[0])
    mflops_increase = (model_df['mflops'].iloc[-1] / model_df['mflops'].iloc[0])
    efficiency = mflops_increase / seq_increase  # Should be > 1 for good scaling
    print(f"   Llama {model}: {seq_increase:.0f}x seq_len → {mflops_increase:.1f}x MFLOPs "
          f"(efficiency: {efficiency:.2f}x)")

print("\n3. MEMORY EFFICIENCY:")
for model in sorted(df['model_size'].unique()):
    model_df = df[df['model_size'] == model]
    mem_per_seq = model_df['total_mem_mb'] / model_df['seq_len']
    print(f"   Llama {model}: {mem_per_seq.mean():.2f} MB per sequence token")

print("\n4. DOMINANT OPERATIONS:")
top_ops_global = ops_df.groupby('operation')['cpu_pct'].mean().sort_values(ascending=False)
for op, pct in top_ops_global.head(3).items():
    print(f"   {op:<20} {pct:>6.1f}% of total CPU time")

print("\n" + "="*70)

## 10. Export High-Quality Plots

In [ ]:
from pathlib import Path

# Create export directory
export_dir = Path('plots')
export_dir.mkdir(exist_ok=True)

# Export 1: MFLOPs scaling
fig, ax = plt.subplots(figsize=(12, 7))
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    ax.plot(model_data['seq_len'], model_data['mflops'], 
           marker='o', linewidth=3, markersize=10, label=f'Llama {model}')
ax.set_xlabel('Sequence Length', fontsize=13, fontweight='bold')
ax.set_ylabel('MFLOPs (Million FLOPs)', fontsize=13, fontweight='bold')
ax.set_title('Transformer Model: Compute Throughput Scaling', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.savefig(export_dir / 'scaling_mflops_hq.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {export_dir / 'scaling_mflops_hq.png'}")

# Export 2: Full comparison dashboard
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# MFLOPs
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    axes[0, 0].plot(model_data['seq_len'], model_data['mflops'], marker='o', label=f'Llama {model}')
axes[0, 0].set_ylabel('MFLOPs')
axes[0, 0].set_title('Compute Throughput')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xscale('log')
axes[0, 0].legend()

# CPU Time
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    axes[0, 1].plot(model_data['seq_len'], model_data['total_cpu_time_s'], marker='s', label=f'Llama {model}')
axes[0, 1].set_ylabel('CPU Time (s)')
axes[0, 1].set_title('Latency')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xscale('log')
axes[0, 1].set_yscale('log')
axes[0, 1].legend()

# Memory
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    axes[1, 0].plot(model_data['seq_len'], model_data['total_mem_mb'], marker='^', label=f'Llama {model}')
axes[1, 0].set_ylabel('Memory (MB)')
axes[1, 0].set_xlabel('Sequence Length')
axes[1, 0].set_title('Memory Usage')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xscale('log')
axes[1, 0].set_yscale('log')
axes[1, 0].legend()

# MFLOPs/sec efficiency
for model in sorted(df['model_size'].unique()):
    model_data = df[df['model_size'] == model].sort_values('seq_len')
    model_data['efficiency'] = model_data['mflops'] / model_data['total_cpu_time_s']
    axes[1, 1].plot(model_data['seq_len'], model_data['efficiency'], marker='d', label=f'Llama {model}')
axes[1, 1].set_ylabel('MFLOPs/sec')
axes[1, 1].set_xlabel('Sequence Length')
axes[1, 1].set_title('Compute Efficiency')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xscale('log')
axes[1, 1].legend()

plt.suptitle('Transformer Model Performance Scaling Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(export_dir / 'performance_dashboard_hq.png', dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved: {export_dir / 'performance_dashboard_hq.png'}")

print(f"\n✓ All plots exported to: {export_dir.absolute()}")

## 11. Generate Summary Report

In [ ]:
# Save summary statistics to file
summary_text = f"""
    TRANSFORMER MODEL PERFORMANCE SCALING ANALYSIS
    {'='*60}
    
    EXECUTIVE SUMMARY
    {'-'*60}
    This analysis evaluates the performance scaling characteristics of LLaMA
    transformer models across different sequence lengths and model sizes.
    
    TEST CONFIGURATION:
    - Device: CPU
    - Batch Size: 1
    - Iterations: {df['batch'].iloc[0]} (averaged)
    - Models: {", ".join(sorted(df['model_size'].unique()))}
    - Sequence Lengths: {", ".join(map(str, sorted(df['seq_len'].unique())))}
    
    KEY METRICS:
    {'-'*60}
    """
    
    for model in sorted(df['model_size'].unique()):
        model_df = df[df['model_size'] == model]
        summary_text += f"""
    Llama {model}:
      - MFLOPs Range: {model_df['mflops'].min():.0f} - {model_df['mflops'].max():.0f}
      - CPU Time Range: {model_df['total_cpu_time_s'].min():.4f}s - {model_df['total_cpu_time_s'].max():.4f}s
      - Memory Range: {model_df['total_mem_mb'].min():.1f}MB - {model_df['total_mem_mb'].max():.1f}MB
      - Avg Efficiency: {(model_df['mflops']/model_df['total_cpu_time_s']).mean():.0f} MFLOPs/sec
        """
    
    summary_text += f"""
    
    SCALING INSIGHTS:
    {'-'*60}
    """
    
    for model in sorted(df['model_size'].unique()):
        model_df = df[df['model_size'] == model].sort_values('seq_len')
        if len(model_df) > 1:
            seq_scale = model_df['seq_len'].iloc[-1] / model_df['seq_len'].iloc[0]
            mflops_scale = model_df['mflops'].iloc[-1] / model_df['mflops'].iloc[0]
            time_scale = model_df['total_cpu_time_s'].iloc[-1] / model_df['total_cpu_time_s'].iloc[0]
            summary_text += f"""
    Llama {model} ({model_df['seq_len'].iloc[0]} → {model_df['seq_len'].iloc[-1]} seq_len):
      - MFLOPs scaling: {mflops_scale:.1f}x
      - Latency scaling: {time_scale:.1f}x
      - Scaling efficiency: {mflops_scale/seq_scale:.2f}x
        """
    
    summary_text += f"""
    
    DOMINANT OPERATIONS:
    {'-'*60}
    """
    
    top_ops = ops_df.groupby('operation')['cpu_pct'].mean().sort_values(ascending=False).head(3)
    for op, pct in top_ops.items():
        summary_text += f"\n    {op:<25} {pct:>6.1f}% of CPU time"
    
    summary_text += f"""
    
    CONCLUSION:
    {'-'*60}
    The transformer models show consistent scaling characteristics with
    increasing sequence length. MFLOPs increase sub-linearly with sequence
    length due to the quadratic complexity of attention mechanisms, while
    memory scales quadratically as expected.
    
    Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
    """
    
    # Save to file
    with open(export_dir / 'ANALYSIS_SUMMARY.txt', 'w') as f:
        f.write(summary_text)
    
    print(summary_text)
    print(f"\n✓ Summary saved to: {export_dir / 'ANALYSIS_SUMMARY.txt'}")